In [1]:
"""
Focal Loss on ReVeal (Debian -> Chrome) with Alpha/Gamma Grid Search
---------------------------------------------------------------------
"""

import os
import json
import time
import random
import traceback
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier


# =========================================================
# Config
# =========================================================
SEED = 42
DATASET_DIR = "../../../../embedding/reveal/codebert/"
SEARCH_RESULTS_PATH = "results/focal_loss_reveal_grid_search.json"
FINAL_RESULTS_PATH  = "results/focal_loss_reveal.json"

ALPHAS = [0.5, 0.65, 0.8, 0.9]
GAMMAS = [0.5, 1.0, 2.0, 3.0]

SEARCH_EPOCHS_PHASE1 = 20
SEARCH_EPOCHS_PHASE2 = 15
SEARCH_ENSEMBLE      = 1

FINAL_EPOCHS_PHASE1 = 50
FINAL_EPOCHS_PHASE2 = 30
FINAL_ENSEMBLE       = 3

os.makedirs(os.path.dirname(SEARCH_RESULTS_PATH), exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Focal Loss
# =========================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, predictions, targets):
        bce     = nn.functional.binary_cross_entropy(
                      predictions, targets, reduction='none'
                  )
        p_t     = predictions * targets + (1 - predictions) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss    = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Result store
# =========================================================
def load_json(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {}


def save_json(path, data):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp_path, path)


def combo_key(alpha, gamma):
    return f"a{alpha}_g{gamma}"


# =========================================================
# Data loading
# =========================================================
def load_data():
    deb_emb = np.load(f"{DATASET_DIR}/debian_embeddings.npy").astype(np.float32)
    deb_lbl = np.load(f"{DATASET_DIR}/debian_labels.npy").astype(np.float32)
    chr_emb = np.load(f"{DATASET_DIR}/chrome_embeddings.npy").astype(np.float32)
    chr_lbl = np.load(f"{DATASET_DIR}/chrome_labels.npy").astype(np.float32)

    scaler  = StandardScaler()
    deb_emb = scaler.fit_transform(deb_emb).astype(np.float32)
    chr_emb = scaler.transform(chr_emb).astype(np.float32)

    n_neg = int(np.sum(deb_lbl == 0))
    n_pos = int(np.sum(deb_lbl == 1))

    print(f"Debian: {deb_emb.shape} (0={n_neg}, 1={n_pos}) | "
          f"Chrome: {chr_emb.shape} (vuln={int(chr_lbl.sum())}) | Device: {DEVICE}")

    return deb_emb, deb_lbl, chr_emb, chr_lbl, n_neg, n_pos


# =========================================================
# Training
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val, optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            predictions = model(x_batch).squeeze(-1)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(x_val).squeeze(-1)
            val_loss = criterion(val_preds, y_val).item()

    return model


def semi_supervised_transfer_learning(x_train, y_train, x_test, criterion, epochs_p1, epochs_p2, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train, test_size=0.2, random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t, optimizer, criterion,
        epochs=epochs_p1, desc=f"Iter {iteration} - Phase 1"
    )

    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze(-1).cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t, optimizer, criterion,
        epochs=epochs_p2, desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze(-1).cpu().numpy()

    return model, y_pred_final


def train_base_model_xgb(x_train, y_train, n_neg, n_pos):
    scale_pos_weight = n_neg / n_pos
    model = XGBClassifier(
        n_estimators=100, max_depth=3, eval_metric="logloss",
        scale_pos_weight=scale_pos_weight
    )
    model.fit(x_train, y_train)
    return model


# =========================================================
# Metrics
# =========================================================
def compute_metrics(y_true, y_pred_proba):
    y_pred = (y_pred_proba > 0.5).astype(int)
    accuracy  = accuracy_score(y_true, y_pred)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    auc       = roc_auc_score(y_true, y_pred_proba)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)
    return {
        "accuracy": round(float(accuracy), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
        "auc": round(float(auc), 4),
        "g_mean": round(float(g_mean), 4),
        "pf": round(float(pf), 4),
    }


# =========================================================
# Run one full experiment
# =========================================================
def run_experiment(deb_emb, deb_lbl, chr_emb, chr_lbl, n_neg, n_pos,
                    alpha, gamma, n_ensemble, epochs_p1, epochs_p2):
    criterion = FocalLoss(alpha=alpha, gamma=gamma)
    ensemble_predictions = np.zeros(len(chr_lbl))

    for i in range(n_ensemble):
        model_nn, _ = semi_supervised_transfer_learning(
            deb_emb, deb_lbl, chr_emb, criterion, epochs_p1, epochs_p2, iteration=i + 1
        )

        model_xgb = train_base_model_xgb(deb_emb, deb_lbl, n_neg, n_pos)
        model_xgb_pred = model_xgb.predict_proba(chr_emb)[:, 1]
        ensemble_predictions += model_xgb_pred

    ensemble_avg = ensemble_predictions / n_ensemble
    return compute_metrics(chr_lbl, ensemble_avg)


# =========================================================
# Stage 1: Grid Search
# =========================================================
def run_grid_search(deb_emb, deb_lbl, chr_emb, chr_lbl, n_neg, n_pos):
    search_results = load_json(SEARCH_RESULTS_PATH)
    print(f"Loaded {len(search_results)} existing grid-search results")

    combos = [(a, g) for a in ALPHAS for g in GAMMAS]
    total = len(combos)

    for idx, (alpha, gamma) in enumerate(tqdm(combos, desc="Grid search", unit="combo"), start=1):
        key = combo_key(alpha, gamma)
        if key in search_results:
            print(f"  [{idx}/{total}] alpha={alpha}, gamma={gamma}: already done "
                  f"(G-Mean={search_results[key].get('g_mean', 'N/A')}) -- skipping.")
            continue

        print(f"  [{idx}/{total}] alpha={alpha}, gamma={gamma}: running...")
        t0 = time.time()
        try:
            metrics = run_experiment(
                deb_emb, deb_lbl, chr_emb, chr_lbl, n_neg, n_pos,
                alpha=alpha, gamma=gamma,
                n_ensemble=SEARCH_ENSEMBLE,
                epochs_p1=SEARCH_EPOCHS_PHASE1, epochs_p2=SEARCH_EPOCHS_PHASE2,
            )
            metrics["alpha"] = alpha
            metrics["gamma"] = gamma
            metrics["elapsed_sec"] = round(time.time() - t0, 1)
            metrics["status"] = "ok"
            search_results[key] = metrics
            save_json(SEARCH_RESULTS_PATH, search_results)
            print(f"    G-Mean={metrics['g_mean']:.4f} | Recall={metrics['recall']:.4f} "
                  f"| F1={metrics['f1']:.4f} ({metrics['elapsed_sec']}s) -- saved.")

        except Exception as e:
            print(f"    FAILED: {e}")
            traceback.print_exc()
            search_results[key] = {
                "alpha": alpha, "gamma": gamma,
                "status": "failed", "error": str(e),
                "elapsed_sec": round(time.time() - t0, 1),
            }
            save_json(SEARCH_RESULTS_PATH, search_results)

    return search_results


def pick_best_combo(search_results):
    valid = {k: v for k, v in search_results.items() if v.get("status") == "ok"}
    if not valid:
        raise RuntimeError("No successful grid-search combinations found.")
    best_key = max(valid, key=lambda k: valid[k]["g_mean"])
    best = valid[best_key]
    print(f"\nBest combination: alpha={best['alpha']}, gamma={best['gamma']} "
          f"(G-Mean={best['g_mean']:.4f})")
    return best["alpha"], best["gamma"]


# =========================================================
# Stage 2: Confirmation Run
# =========================================================
def run_confirmation(deb_emb, deb_lbl, chr_emb, chr_lbl, n_neg, n_pos, alpha, gamma):
    final_results = load_json(FINAL_RESULTS_PATH)
    key = combo_key(alpha, gamma)

    if key in final_results and final_results[key].get("status") == "ok":
        print(f"Confirmation run for alpha={alpha}, gamma={gamma} already done -- skipping.")
        return final_results[key]

    print(f"\nRunning full confirmation: alpha={alpha}, gamma={gamma}, "
          f"epochs={FINAL_EPOCHS_PHASE1}+{FINAL_EPOCHS_PHASE2}, ensemble={FINAL_ENSEMBLE}")
    t0 = time.time()
    try:
        metrics = run_experiment(
            deb_emb, deb_lbl, chr_emb, chr_lbl, n_neg, n_pos,
            alpha=alpha, gamma=gamma,
            n_ensemble=FINAL_ENSEMBLE,
            epochs_p1=FINAL_EPOCHS_PHASE1, epochs_p2=FINAL_EPOCHS_PHASE2,
        )
        metrics["alpha"] = alpha
        metrics["gamma"] = gamma
        metrics["elapsed_sec"] = round(time.time() - t0, 1)
        metrics["status"] = "ok"
        final_results[key] = metrics
        save_json(FINAL_RESULTS_PATH, final_results)
        print(f"Confirmation result: {metrics}")

    except Exception as e:
        print(f"Confirmation FAILED: {e}")
        traceback.print_exc()
        final_results[key] = {
            "alpha": alpha, "gamma": gamma,
            "status": "failed", "error": str(e),
            "elapsed_sec": round(time.time() - t0, 1),
        }
        save_json(FINAL_RESULTS_PATH, final_results)

    return final_results[key]


# =========================================================
# Main
# =========================================================
def main():
    deb_emb, deb_lbl, chr_emb, chr_lbl, n_neg, n_pos = load_data()

    search_results = run_grid_search(deb_emb, deb_lbl, chr_emb, chr_lbl, n_neg, n_pos)
    best_alpha, best_gamma = pick_best_combo(search_results)
    run_confirmation(deb_emb, deb_lbl, chr_emb, chr_lbl, n_neg, n_pos, best_alpha, best_gamma)

    print("\nDONE.")
    print(f"Grid search log: {SEARCH_RESULTS_PATH}")
    print(f"Final confirmed result: {FINAL_RESULTS_PATH}")


if __name__ == "__main__":
    main()

Debian: (18298, 768) (0=16883, 1=1415) | Chrome: (4436, 768) (vuln=825) | Device: mps
Loaded 0 existing grid-search results


Grid search:   0%|                                                                            | 0/16 [00:00<?, ?combo/s]

  [1/16] alpha=0.5, gamma=0.5: running...



Iter 1 - Phase 1: 100%|█████████████████████████████████████████████████████████████████| 20/20 [00:16<00:00,  1.21it/s]
                                                                                                                        
Grid search:   6%|████▎                                                               | 1/16 [00:55<13:59, 55.98s/combo]

    G-Mean=0.4384 | Recall=0.2158 | F1=0.2548 (56.0s) -- saved.
  [2/16] alpha=0.5, gamma=1.0: running...


...


Iter 1 - Phase 1: 100%|█████████████████████████████████████████████████████████████████| 20/20 [00:17<00:00,  1.15it/s]
                                                                                                                        
Grid search:  94%|██████████████████████████████████████████████████████████████▊    | 15/16 [14:15<00:58, 58.47s/combo]

    G-Mean=0.4384 | Recall=0.2158 | F1=0.2548 (62.2s) -- saved.
  [16/16] alpha=0.9, gamma=3.0: running...



Iter 1 - Phase 1: 100%|█████████████████████████████████████████████████████████████████| 20/20 [00:18<00:00,  1.06it/s]
                                                                                                                        
Grid search: 100%|███████████████████████████████████████████████████████████████████| 16/16 [15:17<00:00, 57.35s/combo]


    G-Mean=0.4384 | Recall=0.2158 | F1=0.2548 (61.7s) -- saved.

Best combination: alpha=0.5, gamma=0.5 (G-Mean=0.4384)

Running full confirmation: alpha=0.5, gamma=0.5, epochs=50+30, ensemble=3


Confirmation result: {'accuracy': 0.7653, 'precision': 0.3112, 'recall': 0.2158, 'f1': 0.2548, 'auc': 0.6046, 'g_mean': 0.4384, 'pf': 0.1091, 'alpha': 0.5, 'gamma': 0.5, 'elapsed_sec': 370.6, 'status': 'ok'}

DONE.
Grid search log: results/focal_loss_reveal_grid_search.json
Final confirmed result: results/focal_loss_reveal.json
